# Speech + face emotion recognition on RAVDESS: actor-wise cross-validation and a fair backbone comparison

This notebook evaluates every system with **actor-wise cross-validation** and compares wav2vec 2.0
with emotion2vec under a **matched protocol**.
The shared code lives in the `cv/` package of the repository; this notebook only configures and
drives it.

**Protocol**
- RAVDESS speech, 7 classes (calm merged into neutral), 16 kHz, at most 5.27 s.
- 6 folds of 4 test actors (2 male + 2 female each). Fold *k* validates on the test actors of fold
  *k+1* (also 2M/2F) and trains on the other 16. Every clip is tested exactly once per seed.
- Every fold trains the five reported speech systems, the FER branch (blendshapes → 376 features →
  `XGBRFClassifier`) and the calibrated late fusion of each speech system with FER.
- **Seeds per system.** Systems that train on cached features (both matched frozen probes, E2V-LP),
  the FER model and their fusions run 3 seeds (42, 43, 44). Systems that need the backbone in the
  loop (W2V2-WLP, whose backbone runs in train mode, and W2V2-LoRA) run seed 42 only, so their seed variance is not measured. Fusions of a
  single-seed system use the seed-42 FER model. Seed 42 runs for every system and fold first; the
  extra seeds follow.
- **LoRA is not evaluated under the matched protocol.** Matching it would remove SpecAugment, which
  the LoRA recipe relies on against overfitting. The fair backbone comparison uses frozen encoders only.
- Checkpoint selection, hyperparameter selection and fusion temperatures use the validation actors
  only.

**Systems**

| id | what it is |
|---|---|
| `w2v_wlp` | W2V2-WLP: frozen wav2vec2, weighted-layer probe (frame MLP, SpecAugment, fixed recipe) |
| `w2v_lora` | W2V2-LoRA: PEFT LoRA wav2vec2 (q/v, r=4, gate + LN + linear, fixed recipe) |
| `e2v_lp` | E2V-LP: frozen emotion2vec, final-layer probe (fixed recipe) |
| `w2v_frozen_matched` / `e2v_frozen_matched` | frozen backbone + **shared probe**, same recipe and grid |

**Matched protocol** (identical for both backbones)
- *Layers*: learned softmax weights over **all** hidden states (wav2vec2-base: CNN/positional
  output + 12 layers = 13; emotion2vec_base: input to the first transformer block + 4 context-encoder
  blocks + 8 blocks = 13, read with forward hooks).
- *Pooling*: each clip runs through the backbone **at its true length**, so no padding ever enters
  either backbone and the mean is over valid frames only.
- *Head*: weighted sum → Linear(768→128) → ReLU → Dropout → Linear(128→7).
- *Backbone mode*: always eval (no internal dropout, LayerDrop or SpecAugment); only head dropout is
  active during training.
- *Augmentation*: none, for either backbone. wav2vec2's SpecAugment masks CNN features inside the
  backbone with a learned mask embedding; emotion2vec's masking (data2vec 2.0 mask tokens) is a
  different mechanism, so in-backbone masking cannot be matched. Masking hidden states after the
  backbone is a different regulariser again, and its channel part is the same as dropout on the
  pooled vector, which the head already has. Turning augmentation off for both keeps the two
  pipelines identical and makes the frozen features deterministic, so they can be cached once.
- *Recipe*: AdamW, cosine schedule, class-weighted CE, batch 8, AMP, weight decay 1e-2, up to 30
  epochs, patience 6. Each backbone gets the same 4-point grid (lr ∈ {1e-3, 3e-4} × dropout ∈
  {0.1, 0.3}), selected per fold and seed by the lowest validation loss.

**Running it**
1. For a quick check first, set `SMOKE_TEST = True` (1 fold, 1 seed, 2 epochs,
   separate output folder; 9 clips per actor). Check the backbone printout: both backbones must report **13 hidden
   states**; the notebook stops if they do not.
2. Run the config cell, then everything top to bottom. Each (fold, seed, system, config) run is
   saved to `RESULTS_DIR` as soon as it finishes; after a Colab restart, run the notebook again and
   finished runs are skipped. The frozen-backbone features are cached in `CACHE_DIR`.
3. The analysis section reads only the saved outputs and can be re-run on its own (or from a shell:
   `python -m cv.analysis --results <RESULTS_DIR> --out results_cv`).


In [1]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q -U torchao peft
    !pip install -q datasets transformers torchaudio soundfile
    !pip install -q funasr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 14.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.6/999.6 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 147.5 MB/s eta 0:00:00
 

## Configuration
Everything that controls the run is in this one cell.

In [2]:
# ── Paths (Google Drive placeholders) ───────────────────────────────────────
COLAB_DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_ROOT           = "/content/drive/My Drive"
# Public copy of the blendshapes: https://drive.google.com/drive/folders/1omlBivvIL68xhqAdRY5lSjJNIO8DaIN7
BLENDSHAPE_DATA_PATH = f"{DRIVE_ROOT}/ravdess-videos/blendshapes"
RESULTS_DIR          = f"{DRIVE_ROOT}/speech project cv/results"    # run outputs (resume reads these)
CACHE_DIR            = f"{DRIVE_ROOT}/speech project cv/cache"      # frozen-backbone features
REPO_DIR             = "/content/fer-ser-fusion"                   # where the repo is cloned on Colab
REPO_URL             = "https://github.com/tharencandi/fer-ser-fusion.git"
REPO_BRANCH          = "cv-fair-comparison"

# ── Data and backbones ──────────────────────────────────────────────────────
RAVDESS_DATASET_ID = "xbgoose/ravdess"
TARGET_SR          = 16_000
MAX_SECONDS        = 5.27
W2V_MODEL_ID       = "facebook/wav2vec2-base"
E2V_MODEL_ID       = "iic/emotion2vec_base"     # not emotion2vec+ (its teacher has seen RAVDESS)
E2V_HUB            = "ms"                       # "ms" (ModelScope) or "hf"
REQUIRE_FULL_ALIGNMENT = True    # every audio clip must have its blendshapes

# ── Cross-validation ────────────────────────────────────────────────────────
N_FOLDS   = 6
FOLD_SEED = 42
FOLDS_TO_RUN = None              # None = all folds, or e.g. ["fold0", "fold1"]
PRIMARY_SEED = 42                # runs for every system first (one complete CV as early as possible)
SEEDS_CACHED = [42, 43, 44]      # systems trained on cached features, FER and their fusions
SEEDS_LIVE   = [42]              # systems with the backbone in the training loop
SYSTEM_SEEDS = {
    "w2v_frozen_matched": SEEDS_CACHED, "e2v_frozen_matched": SEEDS_CACHED, "e2v_lp": SEEDS_CACHED,
    "w2v_wlp": SEEDS_LIVE,              # backbone runs in train mode (dropout/SpecAugment): not cacheable
    "w2v_lora": SEEDS_LIVE,
}
FER_SEEDS = SEEDS_CACHED         # fusions of single-seed systems use the PRIMARY_SEED FER model
SYSTEMS_TO_RUN = [
    "w2v_frozen_matched", "e2v_frozen_matched",   # RQ2: matched frozen probes (cached features)
    "e2v_lp",                                     # emotion2vec final-layer probe (cached features)
    "w2v_wlp", "w2v_lora",                        # RQ3: wav2vec2 pipelines (backbone in the loop)
]
RUN_FER = True                   # FER branch + calibrated fusion with every speech system

# ── Training ────────────────────────────────────────────────────────────────
BATCH_SIZE      = 8
EVAL_BATCH_SIZE = 64
USE_AMP         = True
MATCHED_RECIPE = dict(weight_decay=1e-2, epochs=30, patience=6, use_class_weights=True)
MATCHED_GRID_FROZEN = [dict(learning_rate=lr, dropout=do) for lr in (1e-3, 3e-4) for do in (0.1, 0.3)]
EPOCHS_OVERRIDE = None                          # set an int to cap epochs for every system (testing only)

# ── Analysis ────────────────────────────────────────────────────────────────
RELEASE_GPU_WHEN_DONE = True     # Colab: flush Drive and release the runtime after the last cell (stops billing)
BOOTSTRAP_B  = 10_000
ANALYSIS_DIR = f"{RESULTS_DIR}/analysis"        # copy this folder to results_cv/ in the repo

# ── Smoke test ──────────────────────────────────────────────────────────────
SMOKE_TEST = False
SMOKE_TEST = SMOKE_TEST or os.environ.get("CV_SMOKE_TEST") == "1"
OFFLINE    = os.environ.get("CV_OFFLINE") == "1"   # synthetic data + tiny random backbones (no downloads)

if OFFLINE:
    _w = os.environ["CV_WORKDIR"]
    RAVDESS_DATASET_ID, W2V_MODEL_ID, E2V_MODEL_ID = "synthetic", "tiny-random", "tiny-random"
    BLENDSHAPE_DATA_PATH = f"{_w}/blendshapes_synthetic"
    RESULTS_DIR, CACHE_DIR = f"{_w}/results", f"{_w}/cache"
    ANALYSIS_DIR = f"{RESULTS_DIR}/analysis"
if SMOKE_TEST:
    RESULTS_DIR, CACHE_DIR = RESULTS_DIR + "_smoke", CACHE_DIR + "_smoke"
    ANALYSIS_DIR = f"{RESULTS_DIR}/analysis"
    SMOKE_ALL_FOLDS = os.environ.get("CV_SMOKE_ALL_FOLDS") == "1"   # also exercise the full-CV analysis
    FOLDS_TO_RUN = None if SMOKE_ALL_FOLDS else ["fold0"]
    SEEDS_CACHED = [42, 43] if SMOKE_ALL_FOLDS else [42]
    SYSTEM_SEEDS = {s: (SEEDS_CACHED if v is not SEEDS_LIVE else SEEDS_LIVE) for s, v in SYSTEM_SEEDS.items()}
    FER_SEEDS = SEEDS_CACHED
    EPOCHS_OVERRIDE, BOOTSTRAP_B = 2, 200
    MATCHED_GRID_FROZEN = MATCHED_GRID_FROZEN[:2]
SMOKE_CLIPS_PER_ACTOR = "one per emotion" if SMOKE_TEST else None

MAX_SAMPLES = int(TARGET_SR * MAX_SECONDS)
CFG = dict(W2V_MODEL_ID=W2V_MODEL_ID, E2V_MODEL_ID=E2V_MODEL_ID, BATCH_SIZE=BATCH_SIZE,
           EVAL_BATCH_SIZE=EVAL_BATCH_SIZE, USE_AMP=USE_AMP, MATCHED_RECIPE=MATCHED_RECIPE,
           MATCHED_GRID_FROZEN=MATCHED_GRID_FROZEN,
           EPOCHS_OVERRIDE=EPOCHS_OVERRIDE, REQUIRE_FULL_ALIGNMENT=REQUIRE_FULL_ALIGNMENT,
           PRIMARY_SEED=PRIMARY_SEED, SYSTEM_SEEDS=SYSTEM_SEEDS, FER_SEEDS=FER_SEEDS)
assert all(PRIMARY_SEED in v for v in SYSTEM_SEEDS.values()) and PRIMARY_SEED in FER_SEEDS
assert set(SYSTEMS_TO_RUN) <= set(SYSTEM_SEEDS)
print("seeds per system:", {s: SYSTEM_SEEDS[s] for s in SYSTEMS_TO_RUN}, "| FER:", FER_SEEDS)
print(f"SMOKE_TEST={SMOKE_TEST} OFFLINE={OFFLINE}\nRESULTS_DIR={RESULTS_DIR}\nCACHE_DIR={CACHE_DIR}")

seeds per system: {'w2v_frozen_matched': [42, 43, 44], 'e2v_frozen_matched': [42, 43, 44], 'e2v_lp': [42, 43, 44], 'w2v_wlp': [42], 'w2v_lora': [42]} | FER: [42, 43, 44]
SMOKE_TEST=False OFFLINE=False
RESULTS_DIR=/content/drive/My Drive/speech project cv/results
CACHE_DIR=/content/drive/My Drive/speech project cv/cache


## Setup: Drive, repository code, determinism

In [3]:
if IN_COLAB and not OFFLINE:
    from google.colab import drive
    drive.mount(COLAB_DRIVE_MOUNT_POINT)

# Find the cv/ package: a local checkout (running from the repo) or a fresh clone on Colab.
_candidates = [REPO_DIR, os.path.abspath(".."), os.path.abspath(".")]
_repo = next((d for d in _candidates if os.path.isfile(os.path.join(d, "cv", "__init__.py"))), None)
if _repo is None:
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    _repo = REPO_DIR
if _repo not in sys.path:
    sys.path.insert(0, _repo)
print("cv package from", _repo)

import json, time
import numpy as np
import pandas as pd
import torch

from cv import data as D, folds as FO, models as M, runner as R, analysis as A
from cv.train import enable_determinism
from cv.fer import fer_features

enable_determinism()     # before CUDA is used
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
for d in (RESULTS_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)
R.save_json(f"{RESULTS_DIR}/config_{time.strftime('%Y%m%d-%H%M%S')}.json",
            dict(CFG, FOLDS_TO_RUN=FOLDS_TO_RUN, FOLD_SEED=FOLD_SEED, N_FOLDS=N_FOLDS,
                 SYSTEMS_TO_RUN=SYSTEMS_TO_RUN, RUN_FER=RUN_FER, SMOKE_TEST=SMOKE_TEST, OFFLINE=OFFLINE,
                 E2V_HUB=E2V_HUB, MAX_SECONDS=MAX_SECONDS))

Mounted at /content/drive
cv package from /content/fer-ser-fusion


ERROR:root:Error disabling cv.imshow().
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_cv2.py", line 78, in init_code_callback
    module.imshow,
    ^^^^^^^^^^^^^
AttributeError: module 'cv' has no attribute 'imshow'


device: cuda NVIDIA A100-SXM4-40GB


## Data

Two audio views are built:
- `padded_inputs`: zero-padded to 5.27 s and then normalised by the wav2vec2 feature extractor
  (padding included), the input of the pipeline systems (W2V2-WLP, W2V2-LoRA, E2V-LP);
- `waves`: each clip at its true length, zero-mean / unit-variance over its own samples. Both
  backbones of the matched protocol use these.

In [4]:
from transformers import Wav2Vec2FeatureExtractor

raw = D.load_ravdess(RAVDESS_DATASET_ID)
LABEL2ID, ID2LABEL = D.label_space(raw)
NUM_CLASSES = len(LABEL2ID)
clips_all = D.build_clip_table(raw, LABEL2ID)
print(f"{len(clips_all)} clips | classes: {LABEL2ID}")

clips = clips_all
if SMOKE_CLIPS_PER_ACTOR:
    # one clip per RAVDESS emotion per actor (statement 1, repetition 1, strong; neutral is normal-only),
    # plus the calm/normal clip (exercises the calm/neutral alignment)
    keep = (clips_all["statement"] == D.STATEMENT_MAP[1].lower()) & (clips_all["repetition"] == D.REPETITION_MAP[1].lower()) \
        & ((clips_all["intensity"] == "strong") | clips_all["emotion_raw"].isin(["neutral", "calm"]))
    clips = clips_all[keep].reset_index(drop=True)
    print(f"SMOKE: using {len(clips)} clips ({len(clips) // 24} per actor)")

LABELS = clips_all["label"].values          # indexed by HF row index
clips.to_csv(f"{RESULTS_DIR}/clips.csv", index=False)
R.save_json(f"{RESULTS_DIR}/label_space.json", {"label2id": LABEL2ID, "id2label": ID2LABEL})

sub_raw = raw.select(clips["idx"].tolist())
_w = D.load_waveforms(sub_raw, TARGET_SR, MAX_SAMPLES)
waves = [None] * len(clips_all)
for i, w in zip(clips["idx"], _w):
    waves[i] = w
if W2V_MODEL_ID == M.TINY:   # offline test: same normalisation settings as wav2vec2-base
    extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=TARGET_SR, padding_value=0.0, do_normalize=True)
else:
    extractor = Wav2Vec2FeatureExtractor.from_pretrained(W2V_MODEL_ID)
padded_inputs = np.zeros((len(clips_all), MAX_SAMPLES), dtype=np.float32)
padded_inputs[clips["idx"].values] = D.padded_inputs(_w, extractor, TARGET_SR, MAX_SAMPLES)
waves = [D.matched_normalize(w) if w is not None else None for w in waves]
lens = np.array([len(w) for w in _w]) / TARGET_SR
print(f"durations: min {lens.min():.2f}s  mean {lens.mean():.2f}s  max {lens.max():.2f}s  "
      f"(truncated at {MAX_SECONDS}s: {(lens >= MAX_SECONDS - 1e-6).sum()} clips)")

README.md:   0%|          | 0.00/669 [00:00<?, ?B/s]

data/train-00000-of-00002-94d632c9f1f51b(…): reconstructing file:   0%|          |  0.00B /  167MB            

data/train-00000-of-00002-94d632c9f1f51b(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-bcaf733d4b46d6(…): reconstructing file:   0%|          |  0.00B /  158MB            

data/train-00001-of-00002-bcaf733d4b46d6(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1440 [00:00<?, ? examples/s]

1440 clips | classes: {'angry': 0, 'disgust': 1, 'fearful': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprised': 6}


Decoding audio:   0%|          | 0/1440 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

durations: min 2.94s  mean 3.70s  max 5.27s  (truncated at 5.27s: 1 clips)


## Folds

The fold assignment is written to `RESULTS_DIR/folds.json`. If the repository has
`results_cv/folds.json`, the generated assignment must match it.

In [ ]:
fold_spec = FO.make_folds(N_FOLDS, FOLD_SEED)
FO.validate_folds(fold_spec)
_repo_folds = os.path.join(_repo, "results_cv", "folds.json")
if os.path.exists(_repo_folds) and N_FOLDS == 6:
    assert FO.load_folds(_repo_folds)["folds"] == fold_spec["folds"], "fold assignment differs from results_cv/folds.json"
FO.save_folds(fold_spec, f"{RESULTS_DIR}/folds.json")

folds = [f for f in fold_spec["folds"] if FOLDS_TO_RUN is None or f["fold"] in FOLDS_TO_RUN]
display(pd.DataFrame([{**f, "train": len(f["train"])} for f in fold_spec["folds"]]).set_index("fold"))

## Blendshapes and cross-modal alignment

Each audio clip is joined to the blendshapes of its own recording on (raw emotion, intensity,
statement, repetition, actor). The raw emotion keeps calm and neutral apart: neutral exists only at
normal intensity, so a key built after merging calm into neutral would name both the neutral and the
calm recording of the same statement and repetition. The remaining duplicates are the 01
(audio-video) and 02 (video-only) copies of the same recording; the 01 copy is kept.

In [ ]:
if OFFLINE and not os.path.isdir(BLENDSHAPE_DATA_PATH):
    from cv.testing import write_synthetic_blendshapes
    write_synthetic_blendshapes(clips, BLENDSHAPE_DATA_PATH)

bs_samples = D.load_ravdess_blendshape_data(BLENDSHAPE_DATA_PATH)
aligned, align_report = D.align_blendshapes(clips, bs_samples)
R.save_json(f"{RESULTS_DIR}/alignment.json", align_report)
print(json.dumps(align_report, indent=1))
if REQUIRE_FULL_ALIGNMENT and align_report["n_audio_without_face"]:
    raise ValueError("Some audio clips have no blendshapes")

fer_X = np.full((len(clips_all), 376), np.nan, dtype=np.float32)
fer_X[aligned.index.values] = fer_features(aligned)
del bs_samples

checks = [FO.check_fold(clips, f, aligned.index.values, REQUIRE_FULL_ALIGNMENT) for f in fold_spec["folds"]]
display(pd.DataFrame(checks).set_index("fold"))

## Backbones, frozen-feature caches and parameter counts

Frozen backbones are deterministic in eval mode, so their features are computed once per clip and
cached (`CACHE_DIR`):
- `w2v_states`, `e2v_states`: (clips, 13, 768) mean-pooled hidden states for the matched probe;
- `e2v_last`: the input of E2V-LP, the mean of emotion2vec's final-layer output over all frames of
  the padded clip (backbone in eval mode, no padding mask).

`w2v_wlp` cannot be cached: its frozen wav2vec2 runs in train mode (dropout, LayerDrop and
SpecAugment active), so it is run live.

In [9]:
e2v_pristine = M.load_emotion2vec(E2V_MODEL_ID, device=DEVICE, hub=E2V_HUB)
_enc = e2v_pristine.modality_encoders["AUDIO"]
_n_pre, _n_main = len(_enc.context_encoder.blocks), len(e2v_pristine.blocks)
_e2v_states = 1 + _n_pre + _n_main
_w2v_cfg = M.wav2vec2_config(W2V_MODEL_ID)
_w2v_states = _w2v_cfg.num_hidden_layers + 1
print("=" * 78)
print(f"emotion2vec ({E2V_MODEL_ID}) block structure")
print(f"  CNN feature encoder -> projection -> relative positional conv")
print(f"  context encoder (modality_encoders['AUDIO'].context_encoder.blocks): {_n_pre} AltBlocks")
print(f"  main transformer (model.blocks):                                    {_n_main} AltBlocks")
print(f"  embed_dim {e2v_pristine.cfg.get('embed_dim')} | final norm: {e2v_pristine.norm is not None} | "
      f"extra tokens: {_enc.modality_cfg.num_extra_tokens} | eval mode after load: {not e2v_pristine.training}")
print(f"  attention linears: {sorted({n.split('.')[-1] for n, m in e2v_pristine.named_modules() if isinstance(m, torch.nn.Linear) and '.attn.' in n})}")
print(f"  HIDDEN STATES = 1 (input to first block) + {_n_pre} + {_n_main} = {_e2v_states}")
print(f"wav2vec2 ({W2V_MODEL_ID}): HIDDEN STATES = 1 + {_w2v_cfg.num_hidden_layers} layers = {_w2v_states}")
print("=" * 78)
if E2V_MODEL_ID != M.TINY and _e2v_states != 13:
    raise RuntimeError(f"emotion2vec exposes {_e2v_states} hidden states, expected 13: check the model id/structure")
if W2V_MODEL_ID != M.TINY and _w2v_states != 13:
    raise RuntimeError(f"wav2vec2 exposes {_w2v_states} hidden states, expected 13")


def cached(name, compute):
    path = f"{CACHE_DIR}/{name}.npz"
    need = clips["idx"].values
    if os.path.exists(path):
        z = np.load(path)
        if np.isin(need, z["idx"]).all() and z["model"] == (E2V_MODEL_ID if "e2v" in name else W2V_MODEL_ID):
            full = np.zeros((len(clips_all),) + z["feats"].shape[1:], dtype=np.float32)
            full[z["idx"]] = z["feats"]
            print(f"cache hit: {name} {z['feats'].shape}")
            return full
    t0 = time.time()
    feats = compute(need)
    np.savez(path + ".tmp.npz", feats=feats, idx=need, model=(E2V_MODEL_ID if "e2v" in name else W2V_MODEL_ID))
    os.replace(path + ".tmp.npz", path)
    print(f"computed {name} {feats.shape} in {time.time() - t0:.0f}s")
    full = np.zeros((len(clips_all),) + feats.shape[1:], dtype=np.float32)
    full[need] = feats
    return full


w2v_states_mod = M.Wav2Vec2States(W2V_MODEL_ID).to(DEVICE)
e2v_states_mod = M.Emotion2VecStates(e2v_pristine)     # hooks live on e2v_pristine from here on
caches = {
    "w2v_states": cached("w2v_states", lambda ids: M.compute_pooled_states(w2v_states_mod, [waves[i] for i in ids], DEVICE, "wav2vec2 states")),
    "e2v_states": cached("e2v_states", lambda ids: M.compute_pooled_states(e2v_states_mod, [waves[i] for i in ids], DEVICE, "emotion2vec states")),
    "e2v_last":   cached("e2v_last",   lambda ids: M.emotion2vec_last_layer_features(e2v_pristine, padded_inputs[ids], DEVICE)),
}

backbone_frozen_params = {"w2v": sum(p.numel() for p in w2v_states_mod.parameters()),
                          "e2v": sum(p.numel() for p in e2v_states_mod.parameters())}
del w2v_states_mod, e2v_states_mod, e2v_pristine
if DEVICE == "cuda":
    torch.cuda.empty_cache()

funasr version: 1.4.16.


2026-10-23 10:12:21,268 | INFO    | modelscope_hub.download | Downloading 9 files from iic/emotion2vec_base@master


Downloading:   0%|          | 0/9 [00:00<?, ?file/s]

emotion2vec (iic/emotion2vec_base) block structure
  CNN feature encoder -> projection -> relative positional conv
  context encoder (modality_encoders['AUDIO'].context_encoder.blocks): 4 AltBlocks
  main transformer (model.blocks):                                    8 AltBlocks
  embed_dim 768 | final norm: False | extra tokens: 10 | eval mode after load: True
  attention linears: ['proj', 'qkv']
  HIDDEN STATES = 1 (input to first block) + 4 + 8 = 13
wav2vec2 (facebook/wav2vec2-base): HIDDEN STATES = 1 + 12 layers = 13


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
wav2vec2.masked_spec_embed   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cache hit: w2v_states (1440, 13, 768)
cache hit: e2v_states (1440, 13, 768)
cache hit: e2v_last (1440, 768)


In [10]:
# Trainable parameter counts of every system (same models the runs build).
ctx = R.CVContext(cfg=CFG, device="cpu", clips=clips, labels=LABELS, num_classes=NUM_CLASSES,
                  clip_ids=clips_all["clip_id"].values.astype(str),
                  padded_inputs=padded_inputs, caches=caches,
                  fer_X=fer_X, backbone_frozen_params=backbone_frozen_params, aligned_index=aligned.index.values,
                  store=R.RunStore(RESULTS_DIR))
param_rows = []
for s in SYSTEMS_TO_RUN:
    _, recipe = R.system_grid(s, CFG)[0]
    model, *_rest, frozen_extra = R.build_system(ctx, s, recipe)
    p = M.param_breakdown(model)
    if frozen_extra is not None:
        p["frozen"] += int(frozen_extra)
    param_rows.append({"system": s, "label": R.SYSTEMS[s]["label"], **p})
    del model
params_df = pd.DataFrame(param_rows).set_index("system")
R.save_json(f"{RESULTS_DIR}/params.json", params_df.reset_index().to_dict(orient="records"))
display(params_df)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Wav2Vec2Model does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


,label,layer_weights,weight_gate,lora,head,trainable_total,frozen
system,,,,,,,
w2v_frozen_matched,W2V2 frozen (matched),13,0,0,99335,99348,94370944
e2v_frozen_matched,E2V frozen (matched),13,0,0,99335,99348,93790988
e2v_lp,E2V-LP,0,0,0,99335,99335,93790988
w2v_wlp,W2V2-WLP,13,0,0,99335,99348,94371712
w2v_lora,W2V2-LoRA,13,768,147456,6919,155156,94371712


## Training: all folds × seeds × systems

Seed 42 runs first for every system on every fold (one complete cross-validation as early as
possible); then seeds 43 and 44 for the cached systems and FER. Each run is written to Drive when it
finishes; re-running this cell skips finished runs. For every (fold, seed) the FER branch is
trained, each matched system's grid is run and the configuration with the lowest validation loss
is selected, and that system is fused with FER (temperatures fitted on the validation actors).

In [11]:
ctx.device = DEVICE
t0 = time.time()
R.run_all(ctx, folds, SYSTEMS_TO_RUN, with_fer=RUN_FER, verbose_epochs=False)
print(f"session time {(time.time() - t0) / 60:.1f} min")

══ seed 42: w2v_frozen_matched, e2v_frozen_matched, e2v_lp, w2v_wlp, w2v_lora + FER
[done] fer                  fold4  seed 42 test_acc 0.637
[done] w2v_frozen_matched   fold4  seed 42 lr1e-03_do0.10   val_loss 0.6907 epochs 30 test_acc 0.738 params 99,348 13s
        progress 45/174 runs checked, 1.3 min elapsed this session
[done] w2v_frozen_matched   fold4  seed 42 lr1e-03_do0.30   val_loss 0.7173 epochs 30 test_acc 0.738 params 99,348 12s
        progress 46/174 runs checked, 1.5 min elapsed this session
[done] w2v_frozen_matched   fold4  seed 42 lr3e-04_do0.10   val_loss 1.0282 epochs 30 test_acc 0.637 params 99,348 12s
        progress 47/174 runs checked, 1.7 min elapsed this session
[done] w2v_frozen_matched   fold4  seed 42 lr3e-04_do0.30   val_loss 1.0825 epochs 30 test_acc 0.625 params 99,348 12s
        progress 48/174 runs checked, 1.9 min elapsed this session
[done] e2v_frozen_matched   fold4  seed 42 lr1e-03_do0.10   val_loss 0.8756 epochs 27 test_acc 0.725 params 99,348

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[done] w2v_wlp      fold4  seed 42 fixed            val_loss 0.7834 epochs 25 test_acc 0.637 params 99,348 379s
        progress 54/174 runs checked, 9.3 min elapsed this session


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[done] w2v_wlp      fold5  seed 42 fixed            val_loss 0.9110 epochs 20 test_acc 0.721 params 99,348 304s
        progress 65/174 runs checked, 20.9 min elapsed this session


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[done] w2v_lora        fold5  seed 42 fixed            val_loss 0.7356 epochs 11 test_acc 0.771 params 155,156 228s
        progress 66/174 runs checked, 24.7 min elapsed this session
══ seed 43: w2v_frozen_matched, e2v_frozen_matched, e2v_lp + FER
[done] fer                  fold0  seed 43 test_acc 0.608
[done] w2v_frozen_matched   fold0  seed 43 lr1e-03_do0.10   val_loss 0.8742 epochs 26 test_acc 0.792 params 99,348 11s
        progress 67/174 runs checked, 25.3 min elapsed this session
[done] w2v_frozen_matched   fold0  seed 43 lr1e-03_do0.30   val_loss 0.8656 epochs 30 test_acc 0.779 params 99,348 12s
        progress 68/174 runs checked, 25.5 min elapsed this session
[done] w2v_frozen_matched   fold0  seed 43 lr3e-04_do0.10   val_loss 1.0689 epochs 30 test_acc 0.750 params 99,348 12s
        progress 69/174 runs checked, 25.7 min elapsed this session
[done] w2v_frozen_matched   fold0  seed 43 lr3e-04_do0.30   val_loss 1.1226 epochs 30 test_acc 0.725 params 99,348 12s
        progr

## Analysis (reads only the saved outputs)

Re-runnable without training. Writes CSV/JSON tables, PNG figures and `RESULTS_CV.md` into
`ANALYSIS_DIR`; copy that folder to `results_cv/` in the repository.

In [ ]:
from IPython.display import Image, Markdown

res = A.analyse(RESULTS_DIR, ANALYSIS_DIR, B=BOOTSTRAP_B)
display(res["summary"][["label", "n_seeds", "acc_mean", "acc_std", "f1_mean", "pooled_acc", "pooled_acc_lo", "pooled_acc_hi"]].round(4))
if len(res["comparisons"]):
    display(res["comparisons"][["A", "B", "d_acc", "d_acc_lo", "d_acc_hi", "p_boot_acc_holm", "folds_A_better", "wilcoxon_p", "cohen_dz"]].round(4))
for f in sorted(os.listdir(ANALYSIS_DIR)):
    if f.endswith(".png"):
        display(Image(filename=os.path.join(ANALYSIS_DIR, f)))

In [ ]:
display(Markdown(open(os.path.join(ANALYSIS_DIR, "RESULTS_CV.md")).read()))

In [15]:
# Last cell: make sure everything is on Drive, then release the GPU so the session stops billing.
if IN_COLAB and RELEASE_GPU_WHEN_DONE and not OFFLINE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    runtime.unassign()